Training final model -- XGBOOST Hierarchical with Startegy A

In [8]:
"""
XGBoost — two strategies on model2_dataset.csv
"""

import numpy as np
import pandas as pd
import pickle, os, warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

# ── Helpers ───────────────────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate(y_true, y_pred, label=""):
    r  = rmse(y_true, y_pred)
    m  = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mp = np.mean(np.abs((y_true - y_pred) / np.where(y_true==0,1,y_true))) * 100
    print(f"  {label:<25} RMSE={r:.4f}  MAE={m:.4f}  R²={r2:.4f}  MAPE={mp:.2f}%")
    return {"rmse":r, "mae":m, "r2":r2, "mape":mp}

def train_xgb(X_tr, y_tr, X_te, y_te, params, n_trees, label, n_folds=5):
    """Train XGBoost with OOF CV, return test predictions + metrics + final model."""
    kf      = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof     = np.zeros(len(X_tr))
    test_folds = np.zeros((n_folds, len(X_te)))

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tr)):
        Xtr, Xva = X_tr.iloc[tr_idx], X_tr.iloc[va_idx]
        ytr, yva = y_tr.iloc[tr_idx], y_tr.iloc[va_idx]

        d_tr = xgb.DMatrix(Xtr, label=ytr)
        d_va = xgb.DMatrix(Xva, label=yva)
        m = xgb.train(params, d_tr, num_boost_round=n_trees,
                      evals=[(d_va,"val")], early_stopping_rounds=50,
                      verbose_eval=False)
        oof[va_idx]        = m.predict(d_va)
        test_folds[fold]   = m.predict(xgb.DMatrix(X_te))

        f_rmse = rmse(yva, oof[va_idx])
        print(f"    Fold {fold+1}  RMSE={f_rmse:.4f}  trees={m.best_iteration}")

    y_pred = test_folds.mean(axis=0)
    print(f"  OOF RMSE: {rmse(y_tr, oof):.4f}")
    metrics = evaluate(y_te, y_pred, label)

    # Retrain on full training set
    final = xgb.train(params, xgb.DMatrix(X_tr, label=y_tr),
                      num_boost_round=n_trees, verbose_eval=False)
    return y_pred, metrics, final

# ══════════════════════════════════════════════════════════════════════════════
# 1. LOAD & PREPARE
# ══════════════════════════════════════════════════════════════════════════════
df = pd.read_csv("../data/model2_dataset.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)

TARGET = "target_passengers_waiting"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Stratified split — ensures both high/low stops represented equally
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42,
    stratify=df['stop_type_enc']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42,
    stratify=df.loc[X_temp.index, 'stop_type_enc']
)
print(f"Split — train: {len(X_train)}  val: {len(X_val)}  test: {len(X_test)}")

# Scale
SCALE_COLS = [
    "baseline_crowd","speed_factor","time_bucket_te",
    "hour_sin","hour_cos","dow_sin","dow_cos",
    "stop_x_hour_sin","stop_x_hour_cos","baseline_x_weather"
]
scaler = StandardScaler()
X_train[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
X_val[SCALE_COLS]   = scaler.transform(X_val[SCALE_COLS])
X_test[SCALE_COLS]  = scaler.transform(X_test[SCALE_COLS])

X_tv = pd.concat([X_train, X_val]).reset_index(drop=True)
y_tv = pd.concat([y_train, y_val]).reset_index(drop=True)

# Best XGBoost params from previous run
XGB_PARAMS = {
    "objective":"reg:squarederror","eval_metric":"rmse",
    "seed":42,"verbosity":0,
    "learning_rate":0.01,"max_depth":4,
    "min_child_weight":1,"subsample":0.8,
    "colsample_bytree":1.0,"reg_alpha":0.0,"reg_lambda":1.0,
}
N_TREES = 999

# ── Baseline: plain XGBoost (no augmentation) ────────────────────────────────
print("\n── Baseline XGBoost (no augmentation) ──")
_, metrics_base, xgb_base = train_xgb(
    X_tv, y_tv, X_test, y_test,
    XGB_PARAMS, N_TREES, "Baseline XGB"
)

# ══════════════════════════════════════════════════════════════════════════════
# 2. STRATEGY A — EXTREME OVERSAMPLING
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STRATEGY A — Global XGBoost with extreme oversampling")
print("="*60)

THRESHOLD = y_tv.mean() + 2 * y_tv.std()
mask_ext  = y_tv > THRESHOLD
X_normal, y_normal   = X_tv[~mask_ext], y_tv[~mask_ext]
X_extreme, y_extreme = X_tv[mask_ext],  y_tv[mask_ext]

print(f"Extreme threshold : {THRESHOLD:.1f}")
print(f"Normal rows       : {len(X_normal)}")
print(f"Extreme rows      : {len(X_extreme)}")

# Try three oversample multipliers
best_A_rmse, best_A_mult, best_A_pred, best_A_model = np.inf, None, None, None
for mult in [2, 3, 5]:
    X_up, y_up = resample(X_extreme, y_extreme,
                           replace=True, n_samples=len(X_extreme)*mult,
                           random_state=42)
    X_aug = pd.concat([X_normal, pd.DataFrame(X_up, columns=X_tv.columns)]).reset_index(drop=True)
    y_aug = pd.concat([y_normal, pd.Series(y_up)]).reset_index(drop=True)
    print(f"\n  Oversample {mult}x — train pool: {len(X_aug)} "
          f"(extreme weight: {len(X_up)/len(X_aug)*100:.1f}%)")
    pred, metrics, model = train_xgb(
        X_aug, y_aug, X_test, y_test,
        XGB_PARAMS, N_TREES, f"Strat-A {mult}x"
    )
    if metrics['rmse'] < best_A_rmse:
        best_A_rmse  = metrics['rmse']
        best_A_mult  = mult
        best_A_pred  = pred
        best_A_model = model
        best_A_metrics = metrics

print(f"\nBest oversample multiplier: {best_A_mult}x  →  RMSE={best_A_rmse:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. COMPARISON
# ══════════════════════════════════════════════════════════════════════════════
BASELINE_STACK = 6.6920
XGB_SINGLE     = 6.7221

print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)
RESULTS = {
    "Random Forest":       {"rmse":7.070, "mae":4.790, "r2":0.935},
    "LightGBM":            {"rmse":6.879, "mae":4.695, "r2":0.939},
    "CatBoost":            {"rmse":6.914, "mae":4.713, "r2":0.938},
    "XGBoost":             {"rmse":6.722, "mae":4.579, "r2":0.941},
    "Ensemble (Stack)":    {"rmse":6.692, "mae":4.570, "r2":0.942},
    f"XGB Strat-A {best_A_mult}x": {"rmse":best_A_metrics['rmse'], "mae":best_A_metrics['mae'], "r2":best_A_metrics['r2']},
}
print(f"\n  {'Model':<25} {'RMSE':>8}  {'MAE':>7}  {'R²':>7}")
print(f"  {'-'*52}")
for name, m in RESULTS.items():
    print(f"  {name:<25} {m['rmse']:>8.4f}  {m['mae']:>7.4f}  {m['r2']:>7.4f}")
    

# MAE by stop type
print("\n── MAE by stop type ──")
STOP_NAMES = {0:"regular",1:"residential",2:"market",3:"hospital",4:"terminal",5:"university"}
XGB_OLD    = {"regular":1.200,"residential":2.190,"market":3.243,
              "hospital":4.461,"terminal":6.072,"university":8.136}
stop_col   = df.loc[X_test.index,'stop_type_enc'].map(STOP_NAMES)

def mae_by_stop(y_true, y_pred):
    df_e = pd.DataFrame({"stop":stop_col.values,"abs_err":np.abs(y_true.values-y_pred)})
    return df_e.groupby("stop")["abs_err"].mean()

mae_base = mae_by_stop(y_test, xgb_base.predict(xgb.DMatrix(X_test)))
mae_A    = mae_by_stop(y_test, best_A_pred)

print(f"  {'Stop':<14} {'XGB-old':>9}  {'XGB-base':>9}  {'Strat-A':>9}")
for s in ["regular","residential","market","hospital","terminal","university"]:
    old  = XGB_OLD[s]
    base = mae_base.get(s, np.nan)
    a    = mae_A.get(s, np.nan)
    best = min(old, base, a)
    flag_a = " ◄" if a == best and a < old else "  "
    print(f"  {s:<14} {old:>9.3f}  {base:>9.3f}  {a:>9.3f}{flag_a}")
# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATIONS
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Generating plots ──")

BLUE="#378ADD"; CORAL="#D85A30"; AMBER="#EF9F27"
GRAY="#B4B2A9"; LGRAY="#E8E7E1"; TEAL="#1D9E75"; PURPLE="#7F77DD"
STOP_COLORS={"regular":"#9FE1CB","residential":"#5DCAA5","market":"#EF9F27",
             "hospital":"#FAC775","terminal":"#85B7EB","university":"#D85A30"}
stops_ordered = ["regular","residential","market","hospital","terminal","university"]

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":"#D3D1C7",
    "axes.grid":True,"grid.color":"#EBEBEB","grid.linewidth":0.6,
    "font.family":"sans-serif","font.size":10,"axes.titlesize":11,
    "axes.titleweight":"500","axes.labelsize":10,
    "xtick.labelsize":9,"ytick.labelsize":9,
    "axes.spines.top":False,"axes.spines.right":False,
})

fig = plt.figure(figsize=(16, 18))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

model_names  = list(RESULTS.keys())
model_rmses  = [RESULTS[m]["rmse"] for m in model_names]
model_maes   = [RESULTS[m]["mae"]  for m in model_names]
model_r2s    = [RESULTS[m]["r2"]   for m in model_names]
bar_colors   = [LGRAY, "#85B7EB", AMBER, BLUE, PURPLE, TEAL]

# Panel 1: RMSE comparison — all 6 models
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.bar(model_names, model_rmses, color=bar_colors, zorder=3)
ax1.set_ylim(min(model_rmses)*0.97, max(model_rmses)*1.02)
ax1.set_ylabel("RMSE")
ax1.set_title("RMSE — all models")
ax1.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, model_rmses):
    ax1.text(bar.get_x()+bar.get_width()/2, val+0.005,
             f"{val:.4f}", ha="center", fontsize=8.5, color="#5F5E5A")

# Panel 2: R² comparison
ax2 = fig.add_subplot(gs[0, 2])
bars2 = ax2.bar(model_names, model_r2s, color=bar_colors, zorder=3)
ax2.set_ylim(min(model_r2s)*0.998, max(model_r2s)*1.001)
ax2.set_ylabel("R²")
ax2.set_title("R² — all models")
ax2.tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, model_r2s):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.0001,
             f"{val:.4f}", ha="center", fontsize=8, color="#5F5E5A")

# Panel 3: Actual vs Predicted — Strategy A
ax3 = fig.add_subplot(gs[1, :2])
stop_col = df.loc[X_test.index, 'stop_type_enc'].map(STOP_NAMES)
for stop, color in STOP_COLORS.items():
    mask = stop_col.values == stop
    ax3.scatter(y_test.values[mask], best_A_pred[mask],
                color=color, alpha=0.6, s=22, label=stop, edgecolors="none")
lim = max(y_test.max(), best_A_pred.max()) + 5
ax3.plot([0, lim], [0, lim], "--", color=GRAY, linewidth=1.2, label="perfect fit")
ax3.set_xlabel("Actual passengers")
ax3.set_ylabel("Predicted passengers")
ax3.set_title(f"Actual vs predicted — XGB Strat-A {best_A_mult}x")
ax3.legend(fontsize=8, ncol=3, frameon=False, loc="upper left")
ax3.text(0.98, 0.06,
         f"RMSE={best_A_metrics['rmse']:.3f}  MAE={best_A_metrics['mae']:.3f}  R²={best_A_metrics['r2']:.4f}",
         transform=ax3.transAxes, ha="right", va="bottom", fontsize=9, color="#5F5E5A",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="#F1EFE8", edgecolor="#D3D1C7", linewidth=0.6))

# Panel 4: MAE by stop type — pivot
ax4 = fig.add_subplot(gs[1, 2])
XGB_OLD_MAE = {"regular":1.200,"residential":2.190,"market":3.243,
               "hospital":4.461,"terminal":6.072,"university":8.136}
ENSEMBLE_MAE = {"regular":1.128,"residential":2.231,"market":3.269,
                "hospital":4.438,"terminal":6.144,"university":8.075}
x = np.arange(len(stops_ordered))
w = 0.28
ax4.bar(x - w/2, [XGB_OLD_MAE[s]  for s in stops_ordered], width=w, color=BLUE,  label="XGBoost",      zorder=3)
ax4.bar(x + w/2, [mae_A[s]        for s in stops_ordered], width=w, color=TEAL,  label=f"Strat-A {best_A_mult}x", zorder=3)
ax4.set_xticks(x)
ax4.set_xticklabels([s[:4] for s in stops_ordered], fontsize=8)
ax4.set_ylabel("MAE")
ax4.set_title("MAE by stop — XGB vs Strat-A")
ax4.legend(fontsize=8, frameon=False)

fig.suptitle(
    f"XGBoost Strategy A ({best_A_mult}x oversample)  |  "
    f"RMSE={best_A_metrics['rmse']:.3f}  MAE={best_A_metrics['mae']:.3f}  R²={best_A_metrics['r2']:.4f}",
    fontsize=12, fontweight="500", y=0.98
)

# Panel 5: Feature importance — XGB baseline
imp_base_raw = xgb_base.get_score(importance_type="gain")
imp_base = pd.Series({c: imp_base_raw.get(c, 0) for c in X_tv.columns})
imp_base = (imp_base / imp_base.sum() * 100).sort_values(ascending=False).head(12)

ax5 = fig.add_subplot(gs[2, 0])
colors_b = [BLUE if i < 3 else "#85B7EB" if i < 6 else LGRAY for i in range(len(imp_base))]
ax5.barh(imp_base.index[::-1], imp_base.values[::-1], color=colors_b[::-1], height=0.65)
ax5.set_xlabel("Importance (%)")
ax5.set_title("Feature importance — XGB baseline")
ax5.set_xlim(0, imp_base.max() * 1.2)
for i, (feat, val) in enumerate(zip(imp_base.index[::-1], imp_base.values[::-1])):
    ax5.text(val + 0.3, i, f"{val:.1f}%", va="center", fontsize=8, color="#5F5E5A")

# Panel 6: Feature importance — Strategy A
imp_A_raw = best_A_model.get_score(importance_type="gain")
imp_A = pd.Series({c: imp_A_raw.get(c, 0) for c in X_tv.columns})
imp_A = (imp_A / imp_A.sum() * 100).sort_values(ascending=False).head(12)

ax6 = fig.add_subplot(gs[2, 1])
colors_a = [TEAL if i < 3 else "#5DCAA5" if i < 6 else LGRAY for i in range(len(imp_A))]
ax6.barh(imp_A.index[::-1], imp_A.values[::-1], color=colors_a[::-1], height=0.65)
ax6.set_xlabel("Importance (%)")
ax6.set_title(f"Feature importance — Strat-A {best_A_mult}x")
ax6.set_xlim(0, imp_A.max() * 1.2)
for i, (feat, val) in enumerate(zip(imp_A.index[::-1], imp_A.values[::-1])):
    ax6.text(val + 0.3, i, f"{val:.1f}%", va="center", fontsize=8, color="#5F5E5A")

# Panel 7: Importance shift — how oversampling changed feature weights
ax7 = fig.add_subplot(gs[2, 2])
all_feats = list(dict.fromkeys(list(imp_base.index) + list(imp_A.index)))
base_vals = [imp_base.get(f, 0) for f in all_feats]
A_vals    = [imp_A.get(f, 0)    for f in all_feats]
deltas    = [a - b for a, b in zip(A_vals, base_vals)]
delta_s   = pd.Series(deltas, index=all_feats).sort_values()
delta_colors = [TEAL if v >= 0 else CORAL for v in delta_s.values]
ax7.barh(delta_s.index, delta_s.values, color=delta_colors, height=0.65)
ax7.axvline(0, color=GRAY, linewidth=1, linestyle="--")
ax7.set_xlabel("Importance change (%)")
ax7.set_title("Importance shift — baseline → Strat-A")
for i, (feat, val) in enumerate(zip(delta_s.index, delta_s.values)):
    sign = "+" if val >= 0 else ""
    ax7.text(val + (0.1 if val >= 0 else -0.1), i,
             f"{sign}{val:.1f}%", va="center",
             ha="left" if val >= 0 else "right",
             fontsize=7.5, color="#5F5E5A")

save_dir = "../models/model2/xgboost_hierarchical"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(f"{save_dir}/xgb_strategies.png",
            dpi=150, bbox_inches="tight", facecolor="white")
print("  Saved: xgb_strategies.png")
plt.show()

# ── Save ──────────────────────────────────────────────────────────────────────
with open(f"{save_dir}/xgb_stratA_{best_A_mult}x.pkl", "wb") as f:
    pickle.dump(best_A_model, f)

with open(f"{save_dir}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"Saved: xgb_stratA_{best_A_mult}x.pkl")
print(f"Saved: scaler.pkl")

Split — train: 3134  val: 672  test: 672

── Baseline XGBoost (no augmentation) ──
    Fold 1  RMSE=6.3682  trees=753
    Fold 2  RMSE=6.6382  trees=996
    Fold 3  RMSE=6.5983  trees=955
    Fold 4  RMSE=6.3182  trees=741
    Fold 5  RMSE=6.4076  trees=801
  OOF RMSE: 6.4674
  Baseline XGB              RMSE=6.7430  MAE=4.5354  R²=0.9379  MAPE=13.94%

STRATEGY A — Global XGBoost with extreme oversampling
Extreme threshold : 87.3
Normal rows       : 3618
Extreme rows      : 188

  Oversample 2x — train pool: 3994 (extreme weight: 9.4%)
    Fold 1  RMSE=6.3667  trees=998
    Fold 2  RMSE=6.2248  trees=995
    Fold 3  RMSE=6.7683  trees=997
    Fold 4  RMSE=6.4087  trees=994
    Fold 5  RMSE=6.5279  trees=995
  OOF RMSE: 6.4618
  Strat-A 2x                RMSE=6.5601  MAE=4.4148  R²=0.9412  MAPE=13.67%

  Oversample 3x — train pool: 4182 (extreme weight: 13.5%)
    Fold 1  RMSE=6.5162  trees=997
    Fold 2  RMSE=6.8349  trees=994
    Fold 3  RMSE=6.7320  trees=997
    Fold 4  RMSE=6.9380 